# 03 - Evaluación del modelo

**Prerequisito:** haber ejecutado `02_modelo_fraude.ipynb` y tener `models/fraud_model.pkl`.

**Objetivo:** evaluar en profundidad el modelo entrenado con SHAP, curva ROC, análisis por umbral y falsos positivos. Las métricas y gráficas de este notebook son las que van en la documentación del reto (sección 19).

## 1. Setup

In [ ]:
# --- Solo para Google Colab ---
# !git clone https://github.com/Shermy143/fraudia-claims.git
# %cd fraudia-claims
# !pip install -q xgboost shap scikit-learn pandas numpy joblib matplotlib openpyxl

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)

from features.build_features import construir_features, obtener_matriz_modelo, FEATURES_MODELO

print('Setup OK')

## 2. Cargar modelo y datos

In [ ]:
modelo = joblib.load('../models/fraud_model.pkl')
print('Modelo cargado:', type(modelo).__name__)

df = pd.read_csv('../data/processed/siniestros_merged.csv')
print(f'Dataset: {len(df)} registros | {len(df.columns)} columnas')

In [ ]:
# Reproducir el mismo split que en el notebook 02
df_features = construir_features(df)
X = obtener_matriz_modelo(df_features)
y = df_features['etiqueta_fraude_simulada']

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

y_proba = modelo.predict_proba(X_test)[:, 1]
print(f'Test set: {len(X_test)} registros | Fraude real: {y_test.sum()} ({y_test.mean():.1%})')

## 3. Métricas por umbral

Comparamos el umbral por defecto (0.5) contra el ajustado al negocio (0.3).
En seguros el recall es más importante que el precision: es peor dejar pasar un fraude que investigar un caso de más.

In [ ]:
for umbral in [0.5, 0.3]:
    y_pred = (y_proba >= umbral).astype(int)
    print(f'\n=== Umbral {umbral} ===')
    print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraude']))

print(f'AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}')

## 4. Matriz de confusión

In [ ]:
UMBRAL = 0.3
y_pred = (y_proba >= UMBRAL).astype(int)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im)

etiquetas = ['Normal', 'Fraude']
ax.set_xticks([0, 1]); ax.set_xticklabels(etiquetas)
ax.set_yticks([0, 1]); ax.set_yticklabels(etiquetas)
ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
ax.set_title(f'Matriz de confusión (umbral={UMBRAL})')

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i][j]), ha='center', va='center',
                color='white' if cm[i][j] > cm.max() / 2 else 'black', fontsize=14)

plt.tight_layout()
plt.savefig('../docs/matriz_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nVerdaderos negativos  (TN): {cm[0][0]}')
print(f'Falsos positivos      (FP): {cm[0][1]}')
print(f'Falsos negativos      (FN): {cm[1][0]}')
print(f'Verdaderos positivos  (TP): {cm[1][1]}')

## 5. Curva ROC

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.4f}')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1, label='Aleatorio')
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos')
ax.set_title('Curva ROC')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../docs/curva_roc.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Curva Precision-Recall

Más informativa que la ROC cuando el dataset está desbalanceado.

In [ ]:
precision, recall, umbrales = precision_recall_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recall, precision, color='darkorange', lw=2)
ax.axvline(x=recall[(umbrales >= 0.3).argmax()], color='red',
           linestyle='--', lw=1, label='Umbral 0.3')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Curva Precision-Recall')
ax.legend()
plt.tight_layout()
plt.savefig('../docs/curva_precision_recall.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Importancia de features con SHAP

SHAP explica cuánto aporta cada feature a la predicción de cada caso individual. Es la base de `explain_score.py`.

In [ ]:
explainer = shap.TreeExplainer(modelo)
shap_values = explainer.shap_values(X_test)

# Importancia global — qué features pesan más en todo el conjunto de test
plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values, X_test, plot_type='bar',
                  feature_names=FEATURES_MODELO, show=False)
plt.title('Importancia global de features (SHAP)')
plt.tight_layout()
plt.savefig('../docs/shap_importancia_global.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Beeswarm — muestra dirección e intensidad del impacto de cada feature
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_test,
                  feature_names=FEATURES_MODELO, show=False)
plt.title('Impacto de features por dirección (SHAP beeswarm)')
plt.tight_layout()
plt.savefig('../docs/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Explicación de un caso individual

Esto es exactamente lo que hace `explain_score.py` en producción para generar el texto de alerta.

In [ ]:
# Tomar el caso de mayor probabilidad de fraude en el test set
idx_mayor_riesgo = y_proba.argmax()
print(f'Caso más sospechoso — índice en test: {idx_mayor_riesgo}')
print(f'Probabilidad ML: {y_proba[idx_mayor_riesgo]:.4f}')
print(f'Etiqueta real:   {y_test.iloc[idx_mayor_riesgo]}')

# Force plot — qué features empujaron el score hacia arriba o abajo
shap.force_plot(
    explainer.expected_value,
    shap_values[idx_mayor_riesgo],
    X_test.iloc[idx_mayor_riesgo],
    feature_names=FEATURES_MODELO,
    matplotlib=True,
    show=False,
)
plt.title(f'Explicación SHAP — caso {idx_mayor_riesgo}')
plt.tight_layout()
plt.savefig('../docs/shap_caso_ejemplo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top 5 features con mayor impacto para este caso
shap_caso = shap_values[idx_mayor_riesgo]
impacto = pd.DataFrame({
    'feature':  FEATURES_MODELO,
    'valor':    X_test.iloc[idx_mayor_riesgo].values,
    'shap':     shap_caso,
}).reindex(pd.Series(shap_caso).abs().sort_values(ascending=False).index)

print('Top features que determinaron este score:')
print(impacto.head(5).to_string(index=False))

## 9. Análisis de falsos positivos

Casos normales que el modelo marcó como sospechosos. Importante para documentar limitaciones (sección 17 del reto).

In [ ]:
y_pred_03 = (y_proba >= 0.3).astype(int)

mascara_fp = (y_pred_03 == 1) & (y_test.values == 0)
mascara_fn = (y_pred_03 == 0) & (y_test.values == 1)

print(f'Falsos positivos (FP): {mascara_fp.sum()} — casos normales marcados como sospechosos')
print(f'Falsos negativos (FN): {mascara_fn.sum()} — fraudes que no fueron detectados')
print()

# Probabilidades de los FP — ¿están cerca del umbral o muy arriba?
probas_fp = y_proba[mascara_fp]
if len(probas_fp) > 0:
    print(f'Probabilidades de los FP:')
    print(f'  Promedio: {probas_fp.mean():.3f}')
    print(f'  Máximo:   {probas_fp.max():.3f}')
    print(f'  Mínimo:   {probas_fp.min():.3f}')
    print()
    print('Distribución de FP por rango de probabilidad:')
    rangos = pd.cut(probas_fp, bins=[0.3, 0.4, 0.5, 0.6, 0.7, 1.0])
    print(rangos.value_counts().sort_index().to_string())

## 10. Resumen ejecutivo de métricas

Bloque para copiar directo a `docs/uso_ia.md`.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

y_pred_03 = (y_proba >= 0.3).astype(int)
y_pred_05 = (y_proba >= 0.5).astype(int)

print('=' * 50)
print('RESUMEN DE MÉTRICAS DEL MODELO')
print('=' * 50)
print(f'Algoritmo:        XGBoost (clasificación binaria)')
print(f'Dataset:          {len(df)} registros | {y.sum()} fraudes ({y.mean():.1%})')
print(f'Split:            80% train / 20% test (estratificado)')
print(f'Features:         {len(FEATURES_MODELO)}')
print()
print(f'AUC-ROC:          {roc_auc_score(y_test, y_proba):.4f}')
print()
print(f'--- Umbral 0.5 (por defecto) ---')
print(f'  Precision:      {precision_score(y_test, y_pred_05):.4f}')
print(f'  Recall:         {recall_score(y_test, y_pred_05):.4f}')
print(f'  F1:             {f1_score(y_test, y_pred_05):.4f}')
print()
print(f'--- Umbral 0.3 (ajustado al negocio) ---')
print(f'  Precision:      {precision_score(y_test, y_pred_03):.4f}')
print(f'  Recall:         {recall_score(y_test, y_pred_03):.4f}')
print(f'  F1:             {f1_score(y_test, y_pred_03):.4f}')
print()
print('Limitación principal: el modelo fue entrenado con datos')
print('sintéticos. El rendimiento en datos reales puede variar.')
print('Siempre se requiere revisión humana antes de cualquier decisión.')
print('=' * 50)

---

**Próximo paso:** con el modelo validado, el siguiente módulo es `src/explainability/explain_score.py`, que usa los valores SHAP para generar explicaciones en texto legible por el analista.